# Misconception #19: this 95% confidence interval has a 95% chance of containing the effect

**Repeat the same valid study many times. What does 95% confidence mean across the resulting intervals?**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams.update({"figure.dpi": 130, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
BLUE, GRAY, ORANGE = "#236A9F", "#7A828C", "#BF650F"


## A simple model

Assume independent observations from a normal population with known standard deviation $\sigma$. For each simulated study, estimate the mean and form the usual two-sided 95% z interval:

$$\bar X \pm 1.96\,\frac{\sigma}{\sqrt{n}}.$$

The true mean is fixed throughout; the sample and its interval change from study to study.

In [ ]:
TRUE_MEAN = 2.0
SIGMA = 1.0
N = 25
N_STUDIES = 10_000
CONFIDENCE = 0.95
SEED = 1919

rng = np.random.default_rng(SEED)
se = SIGMA / np.sqrt(N)
critical = norm.ppf(1 - (1 - CONFIDENCE) / 2)
estimates = rng.normal(loc=TRUE_MEAN, scale=se, size=N_STUDIES)
lower = estimates - critical * se
upper = estimates + critical * se
covers = (lower <= TRUE_MEAN) & (TRUE_MEAN <= upper)
coverage = covers.mean()
mc_se = np.sqrt(CONFIDENCE * (1 - CONFIDENCE) / N_STUDIES)

print(f"Studies simulated: {N_STUDIES:,}")
print(f"Intervals containing the fixed true mean: {covers.sum():,} ({coverage:.1%})")
print(f"Expected long-run coverage: {CONFIDENCE:.0%} (Monte Carlo SE about {mc_se:.1%})")


## The true effect is fixed; the intervals vary

Each horizontal segment is one simulated 95% interval. The vertical line marks the fixed true mean. Green intervals cover it; orange intervals miss. The display shows a subset so individual intervals remain visible; the coverage calculation uses all simulated studies.

In [ ]:
SHOW = 100
fig, ax = plt.subplots(figsize=(10, 7), layout="constrained")
rows = np.arange(SHOW)
colors = np.where(covers[:SHOW], BLUE, ORANGE)
for y, lo, hi, color in zip(rows, lower[:SHOW], upper[:SHOW], colors):
    ax.plot([lo, hi], [y, y], color=color, alpha=0.75, lw=1.4)
ax.axvline(TRUE_MEAN, color="black", ls="--", lw=1.5, label="Fixed true mean")
ax.set(xlabel="Mean (outcome units)", ylabel="Simulated study", title=f"First {SHOW} of {N_STUDIES:,} repeated 95% intervals")
ax.set_yticks([])
ax.legend(frameon=False, loc="lower right")
plt.show()


## Coverage appears over repetitions

The histogram shows how often intervals cover the true mean across repeated studies. The exact coverage under this model is 95%; the simulated fraction will be close, with small Monte Carlo variation.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.4), layout="constrained")
ax.bar(["Cover true mean", "Miss true mean"], [coverage, 1 - coverage], color=[BLUE, ORANGE], width=0.58)
ax.axhline(CONFIDENCE, color="black", ls="--", lw=1.3, label=f"Nominal coverage = {CONFIDENCE:.0%}")
ax.set(ylim=(0, 1), ylabel="Fraction of simulated intervals", title=f"Long-run coverage across {N_STUDIES:,} studies")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.legend(frameon=False, loc="upper right")
plt.show()
